# OperonAware SCVI — Results Exploration
This notebook loads pre-computed outputs from the evaluation scripts.
It does **not** train models or run heavy computation.

Run order:
```
python scripts/train_standard.py
python scripts/train_operon.py
python scripts/compare_models.py
python scripts/evaluate_regulondb.py
```

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scvi

from config import *
from operon_aware_lib import load_neighbor_indices, build_gene_to_idx, load_operons

## Load per-operon results

In [ ]:
res_df = pd.read_csv(OUT_REGULONDB_CSV)
print(f"Total operons evaluated: {len(res_df)}")
res_df.head()

## Strong operons (most reliable ground truth)

In [ ]:
strong = res_df[res_df['confidence'] == 'Strong']
print(f"Strong operons: {len(strong)}")
print(f"  Standard : {strong['r_standard'].mean():.4f}")
print(f"  Operon   : {strong['r_operon'].mean():.4f}")
print(f"  Δ        : {strong['delta'].mean():+.4f}")
print(f"  % improved: {(strong['delta'] > 0).mean()*100:.1f}%")

print("\nTop 10 most improved Strong operons:")
strong.nlargest(10, 'delta')[['operon_name','n_genes','r_standard','r_operon','delta']]

## Lambda sweep comparison

In [ ]:
# Fill in your sweep results here
sweep = pd.DataFrame([
    {'lambda': 0.01,  'mean_delta': 0.0031, 'pct_improved': 52.9, 'contrast_op': None},
    {'lambda': 0.05,  'mean_delta': 0.0038, 'pct_improved': 53.4, 'contrast_op': None},
    {'lambda': 0.1,   'mean_delta': -0.0025,'pct_improved': 49.6, 'contrast_op': None},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0f1117')
for ax, col, ylabel in [
    (axes[0], 'mean_delta',    'Mean Δ intra-operon r'),
    (axes[1], 'pct_improved',  '% operons improved'),
]:
    ax.plot(sweep['lambda'], sweep[col], 'o-', color='#f06292', lw=2, ms=8)
    ax.axhline(0 if col == 'mean_delta' else 50,
               color='white', lw=0.8, ls='--', alpha=0.5)
    ax.set_facecolor('#1a1d27')
    ax.set_xlabel('lambda_val', color='white')
    ax.set_ylabel(ylabel, color='white')
    ax.tick_params(colors='white')
    ax.set_xscale('log')
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2d3a')
fig.suptitle('Lambda Sweep', color='white', fontsize=13)
plt.tight_layout()
plt.show()